In [5]:
import os

emb_dir = os.path.join('..', 'embedding', 'word_embeddings_cleaned')

In [6]:
import numpy as np

def read_vec_file(file_path):
    words = []

    with open(file_path, "r", encoding="utf-8") as f:
        # Read header
        n_vectors, dim = map(int, f.readline().split())
        print(f"Loading {n_vectors} vectors of dimension {dim}...")

        vectors = np.empty((n_vectors, dim), dtype=np.float32)

        for i, line in enumerate(f):
            parts = line.split()
            if len(parts) != dim + 1:
                raise ValueError(
                    f"Line {i + 2}: expected {dim + 1} columns, got {len(parts)}"
                )

            words.append(parts[0])
            vectors[i] = parts[1:]

    if len(words) != n_vectors:
        raise ValueError(
            f"Header says {n_vectors} vectors, but loaded {len(words)}"
        )

    print(f"Final shape: {vectors.shape}")
    return words, vectors

In [7]:
words, vecs = read_vec_file(os.path.join(emb_dir, 'fasttext_snap1_filt.vec'))

Loading 49460 vectors of dimension 100...
Final shape: (49460, 100)


In [8]:
words

['dibattire',
 'roba',
 'constare',
 'regno',
 'prora',
 'nolare',
 'sirre',
 'enciclico',
 'frappongere',
 'francorchamps',
 'realirre',
 'valentia',
 'fisbo',
 'magone',
 'ferroviere',
 'irto',
 'avico',
 'paura',
 'austere',
 'cisnal',
 'vietnamita',
 'comunicare',
 'spezzettamento',
 'preziosissime',
 'piacevolmente',
 'contrasto',
 'ufficialmente',
 'catturato',
 'obelisco',
 'ritiravare',
 'clicre',
 'consenziente',
 'cessato',
 'indeterminare',
 'favoritismo',
 'demenza',
 'sketch',
 'assassinare',
 'anfim',
 'brundage',
 'contegno',
 'stentatamente',
 'affido',
 'divergere',
 'cresta',
 'udito',
 'provenda',
 'ternario',
 'rarefatto',
 'cokeria',
 'nillare',
 'parodia',
 'dinastico',
 'rocambolesco',
 'dibattito',
 'pallottola',
 'zecchino',
 'astuto',
 'nuotare',
 'georgetown',
 'barbierare',
 'rapportare',
 'euristico',
 'assungere',
 'balsamo',
 'frammentario',
 'tender',
 'torvaianico',
 'valere',
 'emersere',
 'sannio',
 'preservazione',
 'demento',
 'stracarico',
 'conser

In [9]:
import numpy as np

def compute_sims(words, vecs):
    similarities = []

    for i, word in enumerate(words):
        if word.startswith("1'"):
            vec = vecs[i]
            cut_word = word.split("'", 1)[1]  # safe split

            # find the matching word
            for j, word1 in enumerate(words):
                if word1 == cut_word:
                    vec1 = vecs[j]

                    # cosine similarity: (a · b) / (||a|| * ||b||)
                    cos_sim = np.dot(vec, vec1) / (np.linalg.norm(vec) * np.linalg.norm(vec1))
                    
                    # store as a tuple: (original word, cut word, similarity)
                    similarities.append((word, cut_word, cos_sim))
                    break  # stop inner loop once found

    return similarities

In [10]:
sims = compute_sims(words, vecs)

small = []
for el in sims:
    if el[2] < 0.5:
        small.append(el)

small

[]

In [11]:
words, vecs = read_vec_file(os.path.join(emb_dir, 'fasttext_snap2_filt.vec'))

sims = compute_sims(words, vecs)

small = []
for el in sims:
    if el[2] < 0.5:
        small.append(el)

small

Loading 47189 vectors of dimension 100...
Final shape: (47189, 100)


[]

In [12]:
words, vecs = read_vec_file(os.path.join(emb_dir, 'fasttext_snap7_filt.vec'))

sims = compute_sims(words, vecs)

small = []
for el in sims:
    if el[2] < 0.5:
        small.append(el)

small

Loading 64187 vectors of dimension 100...
Final shape: (64187, 100)


[]

In [13]:
from pathlib import Path
import numpy as np

def filter_and_rewrite_embeddings(emb_dir, out_dir):
    emb_dir = Path(emb_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for file_path in emb_dir.iterdir():
        if not file_path.is_file():
            continue

        words = []
        vectors = []

        with open(file_path, "r", encoding="utf-8") as f:
            # Check for optional header
            first_line = f.readline().strip()
            header_parts = first_line.split()
            if len(header_parts) == 2 and header_parts[0].isdigit() and header_parts[1].isdigit():
                _, dim = map(int, header_parts)
                has_header = True
            else:
                parts = first_line.split()
                word = parts[0]

                if not any(ch.isdigit() for ch in word):
                    words.append(word)
                    vectors.append(np.array(parts[1:], dtype=np.float32))

                dim = len(parts) - 1
                has_header = False

            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue

                word = parts[0]

                # REMOVE words containing any digit
                if any(ch.isdigit() for ch in word):
                    continue

                vec = np.array(parts[1:], dtype=np.float32)
                words.append(word)
                vectors.append(vec)

        vectors = np.stack(vectors)

        out_file = out_dir / file_path.name
        with open(out_file, "w", encoding="utf-8") as f:
            if has_header:
                f.write(f"{len(words)} {dim}\n")
            for w, v in zip(words, vectors):
                f.write(f"{w} " + " ".join(f"{x:.6f}" for x in v) + "\n")

        print(f"Processed {file_path.name}: {len(words)} vectors saved.")

In [14]:
filter_and_rewrite_embeddings(emb_dir, os.path.join('..', 'embedding', 'word_embeddings_cleaned'))

Processed fasttext_snap9_filt.vec: 52701 vectors saved.
Processed fasttext_snap8_filt.vec: 58410 vectors saved.
Processed fasttext_snap3_filt.vec: 64187 vectors saved.
Processed fasttext_snap2_filt.vec: 47189 vectors saved.
Processed fasttext_snap4_filt.vec: 64631 vectors saved.
Processed fasttext_snap5_filt.vec: 75488 vectors saved.
Processed fasttext_snap1_filt.vec: 49460 vectors saved.
Processed fasttext_snap10_filt.vec: 48872 vectors saved.
Processed fasttext_snap7_filt.vec: 64187 vectors saved.
Processed fasttext_snap6_filt.vec: 82826 vectors saved.
